# Pipeline B: Graphical Lasso-Based Causal Discovery

This notebook implements a Graphical Lasso approach for discovering undirected relationships.
- Uses GraphicalLassoCV for automatic alpha selection
- Estimates precision matrix and converts to partial correlations
- Produces undirected graphs (cross-sectional data)
- Outputs to timestamped artifact folder

## Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# Add utils to path
sys.path.insert(0, '/Users/manav.neema/Documents/Thesis/Experiment 2/Windows Setup/Causal Discovery/DAG discovery algorithms')
from causal_discovery_utils import *

# Graphical Lasso
from sklearn.covariance import GraphicalLassoCV

print("✓ All imports successful")

## Configuration

In [ ]:
# Pipeline configuration
PIPELINE_NAME = "Graphical_Lasso_Based"
METRICS_CSV_PATH = "/Users/manav.neema/Documents/Thesis/Experiment 2/Windows Setup/Causal Discovery/Metrics data/metrics.csv"
ARTIFACTS_BASE = "/Users/manav.neema/Documents/Thesis/Experiment 2/Windows Setup/Causal Discovery/artifacts"

# Data parameters
MAX_RUNS_TO_PIVOT = 65

# GraphicalLasso parameters
USE_CV = True
CV_ALPHAS = np.logspace(-3, 1, 20)
MANUAL_ALPHA = 0.1
PCOR_THRESHOLD = 0.1
MAX_ITER = 1000

# Feature selection parameters
TARGET_FEATURES = 45
CORRELATION_THRESHOLD = 0.95

print(f"Pipeline B Configuration:")
print(f"  - Method: Graphical Lasso-Based")
print(f"  - Training runs: {MAX_RUNS_TO_PIVOT}")
print(f"  - Use CV: {USE_CV}")
print(f"  - Partial Correlation Threshold: {PCOR_THRESHOLD}")
print(f"  - Target Features: {TARGET_FEATURES}")
print(f"  - Metrics CSV: {METRICS_CSV_PATH}")

## Step 1: Load Metrics Data

In [ ]:
print("[Step 1] Loading metrics data...")
metrics_df = load_metrics_csv(METRICS_CSV_PATH)
print(f"Loaded {metrics_df.shape[0]} metric records")
print(f"Date range: {metrics_df['date'].min()} to {metrics_df['date'].max()}")

## Step 2: Transform Metrics to Matrix

In [ ]:
print("[Step 2] Transforming metrics to matrix...")
metrics_matrix = metrics_to_matrix(metrics_df, max_runs=MAX_RUNS_TO_PIVOT, stage=None)
print(f"Matrix shape: {metrics_matrix.shape}")

## Step 3: Preprocess Metrics Matrix

In [ ]:
print("\n[Step 3] Preprocessing metrics matrix...")
scaled, preprocess_meta = preprocess_metrics_matrix(
    metrics_matrix,
    zscore=True,
    feature_sample_ratio=2.0
)
print(f"After preprocessing: {scaled.shape}")

## Step 4: Feature Selection

In [ ]:
print("\n[Step 4] Feature selection...")
final_features, feature_selection_log = sophisticated_feature_selection(
    scaled,
    target_features=TARGET_FEATURES,
    variance_threshold=1e-6,
    correlation_threshold=CORRELATION_THRESHOLD
)

n_samples, n_features = final_features.shape
print(f"\nGraphicalLasso Requirements Check:")
print(f"  Samples: {n_samples}, Features: {n_features}")
print(f"  Sample-to-feature ratio: {n_samples / n_features:.2f}")

## Step 5: Generate Blacklist

In [ ]:
print("\n[Step 5] Generating blacklist...")
metric_cols = final_features.columns.tolist()
HUMAN_PRIOR_BLACKLIST = generate_stage_blacklist(metric_cols)
print(f"  - Blacklist edges: {len(HUMAN_PRIOR_BLACKLIST)}")

## Step 6: Run Graphical Lasso

In [ ]:
print(f"\n[Step 6] Running GraphicalLasso...")

try:
    if USE_CV:
        print(f"  Using CV with alphas: {len(CV_ALPHAS)} values")
        glasso_cv = GraphicalLassoCV(alphas=CV_ALPHAS, max_iter=MAX_ITER, verbose=0, cv=5)
        glasso_cv.fit(final_features.values)
        best_alpha = glasso_cv.alpha_
        precision_matrix = glasso_cv.precision_
        print(f"  ✓ Best alpha selected: {best_alpha:.6f}")
    else:
        print(f"  Using fixed alpha: {MANUAL_ALPHA}")
        from sklearn.covariance import GraphicalLasso
        glasso = GraphicalLasso(alpha=MANUAL_ALPHA, max_iter=MAX_ITER, verbose=0)
        glasso.fit(final_features.values)
        best_alpha = MANUAL_ALPHA
        precision_matrix = glasso.precision_
        print(f"  ✓ GraphicalLasso completed")
except Exception as e:
    print(f"  ERROR: GraphicalLasso failed: {e}")
    precision_matrix = None
    best_alpha = None

## Step 7: Extract Edges from Precision Matrix

In [ ]:
print(f"\n[Step 7] Extracting edges from precision matrix...")

raw_graphical_lasso_edges = []
if precision_matrix is not None:
    col_names = final_features.columns.tolist()
    
    for i in range(len(col_names)):
        for j in range(i + 1, len(col_names)):  # Upper triangle only (undirected)
            partial_corr = precision_matrix[i, j]
            abs_partial_corr = abs(partial_corr)
            
            if abs_partial_corr > PCOR_THRESHOLD:
                raw_graphical_lasso_edges.append({
                    'from': col_names[i],
                    'to': col_names[j],
                    'edge_type': 'undirected',
                    'partial_corr': float(partial_corr),
                    'abs_partial_corr': float(abs_partial_corr),
                    'weight': float(abs_partial_corr),
                    'source': 'glasso_raw'
                })

print(f"GraphicalLasso discovered {len(raw_graphical_lasso_edges)} edges")
if len(raw_graphical_lasso_edges) == 0:
    print("WARNING: No edges discovered by GraphicalLasso!")

## Step 8: Apply Blacklist Filtering

In [ ]:
print(f"\n[Step 8] Applying blacklist filtering...")

selected_feature_set = set(final_features.columns)
filtered_blacklist = [
    (a, b) for a, b in HUMAN_PRIOR_BLACKLIST 
    if a in selected_feature_set and b in selected_feature_set
]
blacklist_set = set(filtered_blacklist)

print(f"Applicable blacklist edges: {len(filtered_blacklist)}")

# Filter edges - for undirected, check both directions
filtered_edges = []
removed_edges = []

for edge in raw_graphical_lasso_edges:
    edge_tuple_forward = (edge['from'], edge['to'])
    edge_tuple_backward = (edge['to'], edge['from'])
    
    if edge_tuple_forward not in blacklist_set and edge_tuple_backward not in blacklist_set:
        filtered_edges.append(edge)
    else:
        removed_edges.append(edge)

print(f"After blacklist:")
print(f"  - Edges kept: {len(filtered_edges)}")
print(f"  - Edges removed: {len(removed_edges)}")

## Step 9: Visualize Raw Graph

In [ ]:
print(f"\n[Step 9] Visualizing raw graph...")
if len(raw_graphical_lasso_edges) > 0:
    G_raw = visualize_skeleton(
        [(e['from'], e['to'], 'undirected') for e in raw_graphical_lasso_edges],
        title="Pipeline B: GraphicalLasso RAW Graph"
    )
    print(f"Raw graph nodes: {G_raw.number_of_nodes()}, edges: {G_raw.number_of_edges()}")
else:
    print("No raw edges to visualize")

## Step 10: Visualize Filtered Graph

In [ ]:
print(f"\n[Step 10] Visualizing filtered graph...")
if len(filtered_edges) > 0:
    G = visualize_skeleton(
        [(e['from'], e['to'], 'undirected') for e in filtered_edges],
        title="Pipeline B: GraphicalLasso Graph (After Blacklist/Whitelist)"
    )
    print(f"Filtered graph nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")
else:
    print("No filtered edges to visualize")

## Step 11: Compute Baseline Statistics

In [ ]:
print(f"\n[Step 11] Computing baseline statistics...")
baseline_stats = compute_baseline_stats(final_features)
print(f"Computed baseline statistics for {len(baseline_stats)} metrics")

## Step 12: Build Adjacency Maps

In [ ]:
print(f"\n[Step 12] Building adjacency maps...")
upstream_map, downstream_map = build_adjacency_maps(filtered_edges, handle_undirected=True)
print(f"Neighbor map: {len(upstream_map)} nodes")

## Step 13: Export Artifacts

In [ ]:
print(f"\n[Step 13] Exporting artifacts...")

# Create timestamped artifact directory
pipeline_path = create_timestamped_artifact_dir(ARTIFACTS_BASE, PIPELINE_NAME)

print(f"\nSaving artifacts:")

# Save core artifacts
artifacts = {
    "pipeline": PIPELINE_NAME,
    "method": "graphical-lasso-based",
    "data_type": "cross-sectional",
    "status": "SUCCESS" if precision_matrix is not None else "FAILED",
    "timestamp": datetime.now().isoformat(),
    "training_runs": MAX_RUNS_TO_PIVOT,
    "preprocess_meta": preprocess_meta,
    "feature_selection_log": feature_selection_log,
    "graphical_lasso_result": {
        "method": "graphical-lasso",
        "alpha": float(best_alpha) if best_alpha is not None else None,
        "use_cv": USE_CV,
        "pcor_threshold": PCOR_THRESHOLD,
        "n_samples": n_samples,
        "n_features": n_features,
        "sample_feature_ratio": n_samples / n_features,
        "raw_edges_found": len(raw_graphical_lasso_edges)
    },
    "edge_stats": {
        "raw_edges": len(raw_graphical_lasso_edges),
        "removed_by_blacklist": len(removed_edges),
        "final_edges": len(filtered_edges),
        "edge_type": "undirected",
        "weight_type": "partial_correlation"
    },
    "raw_graphical_lasso_edges": raw_graphical_lasso_edges,
    "filtered_edges": filtered_edges,
    "blacklist_filtering": {
        "blacklist_edges_applicable": len(filtered_blacklist),
        "edges_removed": [(e['from'], e['to']) for e in removed_edges]
    },
    "final_graph_stats": {
        "total_edges": len(filtered_edges),
        "nodes_with_neighbors": len(upstream_map)
    }
}

# Save JSON artifacts
save_json_artifact(pipeline_path, 'causal_artifacts.json', artifacts)
save_json_artifact(pipeline_path, 'baseline_stats.json', baseline_stats)
save_json_artifact(pipeline_path, 'upstream_map.json', upstream_map)
save_json_artifact(pipeline_path, 'downstream_map.json', downstream_map)

# Save CSV artifacts
save_csv_artifact(pipeline_path, 'causal_metrics_matrix.csv', final_features)

if len(raw_graphical_lasso_edges) > 0:
    raw_edges_df = pd.DataFrame(raw_graphical_lasso_edges)
    if 'abs_partial_corr' in raw_edges_df.columns:
        raw_edges_df = raw_edges_df.sort_values('abs_partial_corr', ascending=False)
    save_csv_artifact(pipeline_path, 'graphical_lasso_raw_edges.csv', raw_edges_df)
    raw_upstream, raw_downstream = build_adjacency_maps(raw_graphical_lasso_edges, handle_undirected=True)
    save_json_artifact(pipeline_path, 'raw_upstream_map.json', raw_upstream)
    save_json_artifact(pipeline_path, 'raw_downstream_map.json', raw_downstream)

if len(filtered_edges) > 0:
    filtered_edges_df = pd.DataFrame(filtered_edges)
    if 'abs_partial_corr' in filtered_edges_df.columns:
        filtered_edges_df = filtered_edges_df.sort_values('abs_partial_corr', ascending=False)
    save_csv_artifact(pipeline_path, 'graphical_lasso_causal_edges.csv', filtered_edges_df)

print(f"\n" + "="*80)
print(f"✓ PIPELINE B COMPLETE — All artifacts saved")
print(f"="*80)
print(f"\nSaved to: {pipeline_path}")
print(f"\nFinal Results:")
print(f"  - GraphicalLasso alpha: {best_alpha:.6f}" if best_alpha else "  - Alpha: Not computed")
print(f"  - RAW edges: {len(raw_graphical_lasso_edges)}")
print(f"  - FILTERED edges: {len(filtered_edges)}")
print(f"  - Removed: {len(removed_edges)}")